# Load dữ liệu

In [2]:
import pandas as pd

path = "./final-round.json"

# Nếu file là 1 list các object JSON
try:
    df = pd.read_json(path)
    print("Đọc JSON dạng array OK")
except ValueError:
    # Nếu là JSON Lines (mỗi dòng 1 object)
    df = pd.read_json(path, lines=True)
    print("Đọc JSON dạng lines OK")

df.head()

Đọc JSON dạng array OK


,review,sentiment
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,positive
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,positive
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,neutral
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,neutral
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,negative


# Hàm preprocessing: giữ như trước + bỏ dấu tiếng Việt

In [3]:
import re
import unicodedata

def clean_review_basic(text):
    """Tiền xử lý cơ bản như bạn yêu cầu: 
    lower, bỏ số, bỏ ký hiệu, thu gọn khoảng trắng
    """
    if not isinstance(text, str):
        return ""

    # viết thường
    text = text.lower()

    # bỏ số
    text = re.sub(r"\d+", " ", text)

    # bỏ ký hiệu, dấu câu (giữ lại chữ, số, khoảng trắng)
    # \w = chữ + số + _ ; \s = khoảng trắng
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # bỏ riêng dấu gạch dưới nếu còn
    text = text.replace("_", " ")

    # thu gọn khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_vietnamese_accents(text):
    """Bỏ dấu tiếng Việt bằng unicodedata"""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    text = unicodedata.normalize('NFC', text)
    return text


def preprocess_review(text):
    # bước 1: clean cơ bản
    text = clean_review_basic(text)
    # bước 2: bỏ dấu tiếng Việt
    text = remove_vietnamese_accents(text)
    return text

In [4]:
df["review_clean"] = df["review"].apply(preprocess_review)
df[["review", "review_clean", "sentiment"]].head(10)

,review,review_clean,sentiment
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,tu hoc bong thay đoi cuoc đoi đen lop hoc tien...,positive
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,truong đh ton đuc thang cong bo nhom nghien cu...,positive
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,đe thi thu tot nghiep thpt mon toan cua thanh ...,neutral
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,so giao duc tphcm len tieng viec giao vien bi ...,neutral
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,nguoi me tran tro truoc gio ghi đon xanh nguye...,negative
5,"Tốt nghiệp loại giỏi dù không biết đọc viết, n...",tot nghiep loai gioi du khong biet đoc viet nu...,negative
6,Màn ‘hỏi xoáy’ bất ngờ của học sinh BRIS với g...,man hoi xoay bat ngo cua hoc sinh bris voi gia...,positive
7,Tổng Bí thư gợi mở miễn phí bữa trưa cho học s...,tong bi thu goi mo mien phi bua trua cho hoc s...,positive
8,Top 10 trường có điểm chuẩn lớp 10 cao nhất TP...,top truong co điem chuan lop cao nhat tphcm ho...,neutral
9,"Ai là người vẽ bản đồ tác chiến Xuân Lộc 1975,...",ai la nguoi ve ban đo tac chien xuan loc sau n...,positive


# Chuẩn hóa nhãn & encode label

In [5]:
# xem các nhãn hiện có
print(df["sentiment"].value_counts())

# chuẩn hóa về chữ thường
df["sentiment"] = df["sentiment"].str.lower().str.strip()

# mã hóa nhãn → số
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["sentiment"])

print("Classes:", le.classes_)  # ví dụ: ['negative' 'neutral' 'positive']
df[["sentiment", "label_id"]].head()

sentiment
neutral     964
negative    756
positive    724
Name: count, dtype: int64
Classes: ['negative' 'neutral' 'positive']


,sentiment,label_id
0,positive,2
1,positive,2
2,neutral,1
3,neutral,1
4,negative,0


# Chia train / test (và optionally valid)

In [6]:
from sklearn.model_selection import train_test_split

X = df["review_clean"].values      # text đã preprocess + bỏ dấu
y = df["label_id"].values          # nhãn dạng số (0,1,2,...)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/72.0 MB 762.6 kB/s eta 0:01:35
   ---------------------------------------- 0.1/72.0 MB 819.2 kB/s eta 0:01:28
   ---------------------------------------- 0.2/72.0 MB 1.4 MB/s eta 0:00:53
   ---------------------------------------- 0.2/72.0 MB 1.4 MB/s eta 0:00:53
   ---------------------------------------- 0.5/72.0 MB 1.8 MB/s eta 0:00:39
   ---------------------------------------- 0.5/72.0 MB 1.8 MB/s eta 0:00:39
   ---------------------------------------- 0.7/72.0 MB 1.8 MB/s eta 0:00:39
   ---------------------------------------- 0.7/72.0 MB 1.8 MB/s eta 0:00:39
   ---------------------------------------- 0.7/72.0 MB 1.8 MB/s eta 0:00:39
   ---------------------------------------- 0.7/72.0 MB 1.8 MB/s eta 0:00:39
   -------------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Vector hóa text bằng TF-IDF

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

num_classes = len(df["label_id"].unique())

xgb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        max_features=None
    )),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        # GIỮ CỐ ĐỊNH các tham số dưới đây
        n_estimators=300,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0
    ))
])

In [10]:
param_grid = {
    "xgb__max_depth": [3, 4, 5],
    "xgb__min_child_weight": [1, 3, 5],
}

In [11]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2
)

In [12]:
grid.fit(X_train, y_train)

print("Best F1-macro (CV):", grid.best_score_)
print("\nBest params:")
for k, v in grid.best_params_.items():
    print(f"{k}: {v}")

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best F1-macro (CV): 0.6571666394565795

Best params:
xgb__max_depth: 3
xgb__min_child_weight: 1


In [13]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report (test):")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 0.6482617586912065

Classification report (test):
              precision    recall  f1-score   support

           0       0.70      0.64      0.66       151
           1       0.61      0.71      0.65       193
           2       0.67      0.58      0.62       145

    accuracy                           0.65       489
   macro avg       0.66      0.64      0.65       489
weighted avg       0.65      0.65      0.65       489


Confusion matrix:
[[ 96  41  14]
 [ 29 137  27]
 [ 13  48  84]]


In [14]:
target_names = le.inverse_transform(sorted(df["label_id"].unique()))
print(classification_report(y_test, y_pred, target_names=target_names))

              precision    recall  f1-score   support

    negative       0.70      0.64      0.66       151
     neutral       0.61      0.71      0.65       193
    positive       0.67      0.58      0.62       145

    accuracy                           0.65       489
   macro avg       0.66      0.64      0.65       489
weighted avg       0.65      0.65      0.65       489



## KNN


In [15]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedKFold

knn_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9)),
    ("knn", KNeighborsClassifier())
])

param_grid_knn = {
    "knn__n_neighbors": [3, 5, 7, 9],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["cosine", "euclidean"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_knn = GridSearchCV(
    knn_pipeline,
    param_grid_knn,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid_knn.fit(X_train, y_train)

print("Best KNN params:", grid_knn.best_params_)
print("Best CV F1-macro:", grid_knn.best_score_)


Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best KNN params: {'knn__metric': 'cosine', 'knn__n_neighbors': 9, 'knn__weights': 'distance'}
Best CV F1-macro: 0.6479363680482279


In [16]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

knn_best = grid_knn.best_estimator_
y_pred_knn = knn_best.predict(X_test)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn, target_names=target_names))
print(confusion_matrix(y_test, y_pred_knn))


KNN Accuracy: 0.6462167689161554
              precision    recall  f1-score   support

    negative       0.67      0.72      0.69       151
     neutral       0.62      0.69      0.66       193
    positive       0.65      0.52      0.58       145

    accuracy                           0.65       489
   macro avg       0.65      0.64      0.64       489
weighted avg       0.65      0.65      0.64       489

[[108  34   9]
 [ 29 133  31]
 [ 24  46  75]]


## Random Forest


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9)),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    ))
])

param_grid_rf = {
    "rf__max_depth": [None, 20, 40],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid_rf.fit(X_train, y_train)

print("Best RF params:", grid_rf.best_params_)
print("Best CV F1-macro:", grid_rf.best_score_)


Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best RF params: {'rf__max_depth': None, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 5}
Best CV F1-macro: 0.6724215214045034


In [18]:
rf_best = grid_rf.best_estimator_
y_pred_rf = rf_best.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=target_names))
print(confusion_matrix(y_test, y_pred_rf))


RF Accuracy: 0.6912065439672802
              precision    recall  f1-score   support

    negative       0.72      0.74      0.73       151
     neutral       0.65      0.76      0.70       193
    positive       0.73      0.55      0.63       145

    accuracy                           0.69       489
   macro avg       0.70      0.68      0.69       489
weighted avg       0.70      0.69      0.69       489

[[111  33   7]
 [ 24 147  22]
 [ 19  46  80]]
